# Tiền xử lý và Chuẩn hóa Dữ liệu Dân số (Data Preprocessing)

Notebook này thực hiện quy trình ETL (Extract - Transform - Load) từ các file thô `data/raw/` thành các bảng dữ liệu sạch trong `data/processed/`:
1. Chuyển đổi sang định dạng dài chuẩn hóa (`population_fact_long.csv`).
2. Tách bạch dữ liệu quan sát lịch sử (`estimate`) và dự báo kịch bản trung bình (`projected`).
3. Tạo bảng định dạng rộng (`population_fact_wide.csv`) phục vụ mô hình hóa và Tableau.
4. Đối chiếu kiểm chứng chéo với nguồn World Population Review (`world_population_review_validation.csv`).
5. Tạo Từ điển dữ liệu (`data_dictionary.csv`) và Báo cáo chất lượng dữ liệu (`data_quality_report.json`).

In [1]:
import csv
import json
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

# Thiết lập đường dẫn thư mục
ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục Raw:", RAW_DIR)
print("Thư mục Processed:", PROCESSED_DIR)

Thư mục Raw: /Users/phitaan/Documents/WORKSPACE/TTDLTQ/PROJECT CUỐI KỲ - BASIC/data/raw
Thư mục Processed: /Users/phitaan/Documents/WORKSPACE/TTDLTQ/PROJECT CUỐI KỲ - BASIC/data/processed


## 1. Đọc và Trích xuất Dữ liệu Nguồn từ UN WPP (OWID)
Quy chuẩn 3 bảng dữ liệu chính: Dân số (Population), Tốc độ tăng trưởng (Population growth rate) và Tổng tỷ suất sinh (Total fertility rate).

In [2]:
# Cấu hình nguồn dữ liệu và các cột chỉ số
SOURCE_SPECS = (
    ("population-with-un-projections.csv", "Population", "people", "Population", "Population (Projected)"),
    ("population-growth-rates.csv", "Population growth rate", "percent", "Population growth rate", "Population growth rate (%) (Projected)"),
    ("children-born-per-woman.csv", "Total fertility rate", "children per woman", "Total fertility rate", None),
)

main_records = []
for filename, indicator, unit, est_col, proj_col in SOURCE_SPECS:
    file_path = RAW_DIR / filename
    with file_path.open(encoding="utf-8", newline="") as f:
        file_rows = list(csv.DictReader(f))
    
    for row in file_rows:
        for col, status in ((est_col, "estimate"), (proj_col, "projected")):
            if col is None:
                continue
            val_str = row.get(col)
            if not val_str or not val_str.strip():
                continue
            val = float(val_str)
            main_records.append({
                "Entity": row["Entity"].strip(),
                "Code": row.get("Code", "").strip() or None,
                "Year": int(row["Year"]),
                "Indicator": indicator,
                "Value": val,
                "Unit": unit,
                "DataStatus": status,
                "Source": "UN WPP 2024 processed by OWID",
                "SourceUrl": "https://ourworldindata.org/grapher/population-with-un-projections",
            })

print(f"Tổng số bản ghi đã trích xuất: {len(main_records):,}")

Tổng số bản ghi đã trích xuất: 97,167


## 2. Khử Trùng lặp Khóa và Xuất Bảng Dài (Long Format)
Khoá chính logic gồm: `(Entity, Code, Year, Indicator, DataStatus)`.

In [3]:
# Khử trùng lặp khóa
unique_records = {}
for r in main_records:
    key = (r["Entity"], r["Code"], r["Year"], r["Indicator"], r["DataStatus"])
    unique_records[key] = r

deduped_records = list(unique_records.values())
dup_count = len(main_records) - len(deduped_records)
print(f"Số bản ghi sau khi khử trùng lặp: {len(deduped_records):,}")
print(f"Số dòng trùng lặp đã loại bỏ: {dup_count}")

# Lưu ra population_fact_long.csv
long_fields = ["Entity", "Code", "Year", "Indicator", "Value", "Unit", "DataStatus", "Source", "SourceUrl"]
with (PROCESSED_DIR / "population_fact_long.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=long_fields)
    writer.writeheader()
    writer.writerows(deduped_records)

print(f"Đã xuất bảng Long Format: {PROCESSED_DIR / 'population_fact_long.csv'}")

Số bản ghi sau khi khử trùng lặp: 97,167
Số dòng trùng lặp đã loại bỏ: 0


Đã xuất bảng Long Format: /Users/phitaan/Documents/WORKSPACE/TTDLTQ/PROJECT CUỐI KỲ - BASIC/data/processed/population_fact_long.csv


## 3. Tạo Bảng Rộng (Wide Format) phục vụ Tableau và Mô hình hóa
Nhóm theo `(Entity, Code, Year, DataStatus)`, trải các Indicator thành các cột riêng biệt.

In [4]:
# Tạo bảng Wide Format
wide_dict = {}
for r in deduped_records:
    key = (r["Entity"], r["Code"], r["Year"], r["DataStatus"])
    if key not in wide_dict:
        wide_dict[key] = {
            "Entity": r["Entity"],
            "Code": r["Code"],
            "Year": r["Year"],
            "DataStatus": r["DataStatus"]
        }
    wide_dict[key][r["Indicator"]] = r["Value"]

wide_rows = list(wide_dict.values())
wide_fields = ["Entity", "Code", "Year", "DataStatus", "Population", "Population growth rate", "Total fertility rate"]

with (PROCESSED_DIR / "population_fact_wide.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=wide_fields)
    writer.writeheader()
    writer.writerows(wide_rows)

print(f"Đã xuất bảng Wide Format: {PROCESSED_DIR / 'population_fact_wide.csv'}")
print(f"Tổng số dòng bảng Wide: {len(wide_rows):,}")

Đã xuất bảng Wide Format: /Users/phitaan/Documents/WORKSPACE/TTDLTQ/PROJECT CUỐI KỲ - BASIC/data/processed/population_fact_wide.csv
Tổng số dòng bảng Wide: 39,798


## 4. Đối chiếu Kiểm chứng Chéo với World Population Review (2024 - 2026)

In [5]:
# Bảng ánh xạ từ đồng nghĩa giữa quy ước WPR và UN WPP
ENTITY_ALIASES = {
    "dr congo": "Democratic Republic of Congo",
    "ivory coast": "Cote d'Ivoire",
    "macau": "Macao",
    "micronesia": "Micronesia (country)",
    "republic of the congo": "Congo",
    "saint helena ascension and tristan da cunha": "Saint Helena",
    "saint martin": "Saint Martin (French part)",
    "sint maarten": "Sint Maarten (Dutch part)",
    "timor leste": "East Timor",
    "vatican city": "Vatican",
}

# Đọc dữ liệu WPR để đối chiếu
def normalize_name(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

entity_map = {normalize_name(str(r["Entity"])): r for r in deduped_records}
for alias_norm, target_name in ENTITY_ALIASES.items():
    target_norm = normalize_name(target_name)
    if target_norm in entity_map:
        entity_map[alias_norm] = entity_map[target_norm]

validation_rows = []
wpr_file = RAW_DIR / "world-population-review-2024-2026.csv"
if wpr_file.exists():
    with wpr_file.open(encoding="utf-8", newline="") as f:
        wpr_data = list(csv.DictReader(f))
    
    for row in wpr_data:
        m = entity_map.get(normalize_name(row["entity"]))
        validation_rows.append({
            "Entity": m["Entity"] if m else row["entity"],
            "Code": m["Code"] if m else None,
            "Year": int(row["year"]),
            "PopulationWPR": float(row["population"]),
            "SourceUrl": row["source_url"],
            "MatchStatus": "matched" if m else "unmatched",
        })

    val_fields = ["Entity", "Code", "Year", "PopulationWPR", "SourceUrl", "MatchStatus"]
    with (PROCESSED_DIR / "world_population_review_validation.csv").open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=val_fields)
        writer.writeheader()
        writer.writerows(validation_rows)
    matched_cnt = sum(1 for r in validation_rows if r["MatchStatus"] == "matched")
    unmatched_cnt = sum(1 for r in validation_rows if r["MatchStatus"] == "unmatched")
    print(f"Đã xuất bảng đối chiếu WPR: {len(validation_rows)} dòng (Khớp: {matched_cnt}, Không khớp: {unmatched_cnt})")
    print(f"Tỷ lệ khớp thực thể: {matched_cnt / len(validation_rows) * 100:.1f}%")


Đã xuất bảng đối chiếu WPR: 705 dòng (Khớp: 705, Không khớp: 0)
Tỷ lệ khớp thực thể: 100.0%


## 5. Xuất Từ điển Dữ liệu và Báo cáo Kiểm tra Chất lượng (Quality Gates)

In [6]:
# 1. Từ điển dữ liệu
dict_rows = [
    {"table_name": "population_fact_long", "column_name": "Entity", "data_type": "string", "description": "Tên quốc gia hoặc thực thể khu vực"},
    {"table_name": "population_fact_long", "column_name": "Code", "data_type": "string", "description": "Mã ISO-3 hoặc mã định danh của OWID/UN"},
    {"table_name": "population_fact_long", "column_name": "Year", "data_type": "integer", "description": "Năm quan sát hoặc năm dự phóng"},
    {"table_name": "population_fact_long", "column_name": "Indicator", "data_type": "string", "description": "Tên chỉ tiêu dân số"},
    {"table_name": "population_fact_long", "column_name": "Value", "data_type": "float", "description": "Giá trị của chỉ tiêu tương ứng"},
    {"table_name": "population_fact_long", "column_name": "Unit", "data_type": "string", "description": "Đơn vị đo lường"},
    {"table_name": "population_fact_long", "column_name": "DataStatus", "data_type": "string", "description": "Phân loại trạng thái: estimate hoặc projected"},
    {"table_name": "population_fact_long", "column_name": "Source", "data_type": "string", "description": "Nguồn cung cấp dữ liệu gốc"},
    {"table_name": "population_fact_long", "column_name": "SourceUrl", "data_type": "string", "description": "Đường dẫn tham chiếu đến dữ liệu gốc"},
]
dict_fields = ["table_name", "column_name", "data_type", "description"]
with (PROCESSED_DIR / "data_dictionary.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=dict_fields)
    writer.writeheader()
    writer.writerows(dict_rows)

# 2. Báo cáo chất lượng dữ liệu
indicator_counts = Counter(r["Indicator"] for r in deduped_records)
status_counts = Counter(r["DataStatus"] for r in deduped_records)
years = [r["Year"] for r in deduped_records]

quality_report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "row_count": len(deduped_records),
    "entity_count": len({r["Entity"] for r in deduped_records}),
    "year_min": min(years),
    "year_max": max(years),
    "duplicate_count_removed": dup_count,
    "missing_key_count": sum(1 for r in deduped_records if not r["Entity"] or not r["Year"] or not r["Indicator"]),
    "indicator_counts": dict(indicator_counts),
    "status_counts": dict(status_counts),
    "wpr_validation_rows": len(validation_rows),
    "wpr_unmatched_rows": sum(1 for r in validation_rows if r["MatchStatus"] == "unmatched"),
    "quality_gates": {
        "at_least_5000_rows": len(deduped_records) >= 5000,
        "at_least_3_indicators": len(indicator_counts) >= 3,
        "no_missing_keys": sum(1 for r in deduped_records if not r["Entity"] or not r["Year"] or not r["Indicator"]) == 0,
    }
}

(PROCESSED_DIR / "data_quality_report.json").write_text(
    json.dumps(quality_report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Đã xuất data_dictionary.csv và data_quality_report.json thành công!")
print("Kiểm tra Quality Gates:", quality_report["quality_gates"])

Đã xuất data_dictionary.csv và data_quality_report.json thành công!
Kiểm tra Quality Gates: {'at_least_5000_rows': True, 'at_least_3_indicators': True, 'no_missing_keys': True}
